<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/2_%EA%B8%B0%EA%B3%84%EB%8F%85%ED%95%B4_%EC%A7%88%EB%AC%B8%EC%9D%84_%EA%B7%B8%EB%8C%80%EB%A1%9C_%EC%82%AC%EC%9A%A9%ED%95%98%EB%8A%94_%EB%8D%B0%EC%9D%B4%ED%84%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install datasets openai

#데이터 불러오기

네거티브 샘플링 에서 만든 데이터셋 불러오기

In [2]:
from datasets import load_dataset

dataset = load_dataset("iamjoon/klue-mrc-bge-m3")
df = dataset['train'].to_pandas()
df = df[:100]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/640 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/77.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10434 [00:00<?, ? examples/s]

In [3]:
sample = df.iloc[0]

print(f'question : {sample.question}')
print(f'title : {sample.title}')
print(f'context : {sample.context}')
print(f'negative_samples : {sample.negative_samples}')
print(f'answer_text : {sample.answer_text}')

question : 북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?
title : 제주도 장마 시작 … 중부는 이달 말부터
context : 올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도 늦은 이달 말께 장마가 시작될 전망이다.17일 기상청에 따르면 제주도 남쪽 먼바다에 있는 장마전선의 영향으로 이날 제주도 산간 및 내륙지역에 호우주의보가 내려지면서 곳곳에 100㎜에 육박하는 많은 비가 내렸다. 제주의 장마는 평년보다 2~3일, 지난해보다는 하루 일찍 시작됐다. 장마는 고온다습한 북태평양 기단과 한랭 습윤한 오호츠크해 기단이 만나 형성되는 장마전선에서 내리는 비를 뜻한다.장마전선은 18일 제주도 먼 남쪽 해상으로 내려갔다가 20일께 다시 북상해 전남 남해안까지 영향을 줄 것으로 보인다. 이에 따라 20~21일 남부지방에도 예년보다 사흘 정도 장마가 일찍 찾아올 전망이다. 그러나 장마전선을 밀어올리는 북태평양 고기압 세력이 약해 서울 등 중부지방은 평년보다 사나흘가량 늦은 이달 말부터 장마가 시작될 것이라는 게 기상청의 설명이다. 장마전선은 이후 한 달가량 한반도 중남부를 오르내리며 곳곳에 비를 뿌릴 전망이다. 최근 30년간 평균치에 따르면 중부지방의 장마 시작일은 6월24~25일이었으며 장마기간은 32일, 강수일수는 17.2일이었다.기상청은 올해 장마기간의 평균 강수량이 350~400㎜로 평년과 비슷하거나 적을 것으로 내다봤다. 브라질 월드컵 한국과 러시아의 경기가 열리는 18일 오전 서울은 대체로 구름이 많이 끼지만 비는 오지 않을 것으로 예상돼 거리 응원에는 지장이 없을 전망이다.
negative_samples : ['궤도물리학은 계절의 지속 기간이 지구의 궤도가 지점과 분점 사이의 공간을 휩쓸고 지나가는 면적이 클수록 길어지며, 따라서 만약 이심률이 극단적으로 커진다면 원일점 쪽에서 나타나는 계절이 오래 지속될 것이다고 예측한다. 현재 지구에서는, 지구가 근일점에 접근할수록(태양에 가까워질수록) 북반구

#데이터 정제

검색 결과 (pos + neg) 섞기

In [4]:
import random

# 정답이 첫번째 문서에만 위치하지 않도록 문서를 섞는다.
samples = []
for context, negative_samples in zip(df['context'].to_list(), df['negative_samples'].to_list()):
  documents = [context] + list(negative_samples)
  random.shuffle(documents)
  samples.append(documents)

#LLM 사용

LLM 사용해 질문에 대한 답 만들기???????

user prompt, system prompt 만들기

In [5]:
#user_prompt 생성 - 질문 + 검색결과 (pos, 4 neg)
user_prompts = []
for sample in samples:
  user_prompt = '검색 결과:\n'
  for idx, s in enumerate(sample):
    user_prompt = user_prompt  + '문서' + str(idx + 1) + ':' + s + '\n' + '---\n'
  user_prompts.append(user_prompt)

user_prompt_for_answer = []
for question, user_prompt in zip(df['question'].to_list(), user_prompts):
  user_prompt_for_answer.append('질문:' + question + '\n' + user_prompt + '답변:')

In [6]:
#system_prompt 생성
system_prompt = """당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문으로부터 주어진 다수의 문서에서 답을 찾아 작성하세요.
2. 검색된 문서에 질문의 답이 없는 경우에는 질문에 대한 내용을 언급하면서 답을 찾을 수 없다고 작성하세요.
3. 답변에서 참고자료의 내용을 인용한 경우, 답변 맨 끝에 인용한 문서 id를 추가하세요. 답변에 내용을 인용한 경우에만 작성해야 합니다. 인용한 게 없다면 쓰지마세요.
4. 문서 id 값은 검색 결과에서 가져온 것이며 두 개의 대괄호로 묶어야 하며 ref라는 키워드와 함께 작성해야 합니다. 예를 들어 특정 문단이 3번, 5번 문서에서 인용했다면 다음과 같이 작성하십시오.
예시) ~~~해서 했습니다. [[ref3]], [[ref5]]
5. 주요 문장 또는 문단 뒤에 ref문서번호를 덧붙이는 것은 매우 중요합니다. 이는 반드시 지켜져야 합니다.
6. 문단 중간 중간에 인용이 들어가더라도 답변은 하나의 이어지는 평문으로 작성하십시오.
7. 주어진 문서는 검색 결과입니다. 따라서 질문과 연관없는 문서가 숨어있을 수 있으니 이를 인용하지 않도록 주의하십시오.
8. 검색 결과에 없는 내용은 답변으로 작성하지 마십시오. 가장 중요한 지시사항입니다. 따라서 인용 없이는 그 어떠한 설명도 제공할 수 없습니다.
9. 답변은 최대한 풍부하게 작성해주시기 바랍니다.
10. 검색 결과는 모두 독립적인 문서들이며 반드시 연관되어져 있거나 순서대로 이어지는 문서들이 아닙니다.
"""

In [32]:
from tqdm import tqdm
import openai

client = openai.OpenAI(api_key="")

In [10]:
# 5개로 테스트
result_lst = []
for user_prompt in tqdm(user_prompt_for_answer[:10]):
  response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_prompt}
      ],
    temperature=0
  )
  result_lst.append(response.choices[0].message.content)

100%|██████████| 10/10 [00:32<00:00,  3.26s/it]


#파싱

In [11]:
import re

# 인용 문서 번호 추출
def extract_ref_numbers(text):
    # 정규표현식을 사용하여 [[ref숫자]] 패턴 찾기
    pattern = r'\[\[ref(\d+)\]\]'

    # 모든 매치 찾기
    matches = re.findall(pattern, text)

    # 문자열을 정수로 변환하고 중복 제거 후 정렬
    unique_numbers = sorted(set(map(int, matches)))

    return unique_numbers

In [14]:
extracted_ref_numbers = []
for sample in result_lst:
  extracted_ref_numbers.append(extract_ref_numbers(sample))

In [31]:
import pandas as pd

result_df = pd.DataFrame({
    'question' : df['question'][:10],
    'search_result' : samples[:10],
    'answer' : result_lst,
    'extracted_ref_numbers' : extracted_ref_numbers
})

result_df = result_df[~(result_df['extracted_ref_numbers'].apply(str) == '[]')].reset_index(drop=True)

result_df.head()

,question,search_result,answer,extracted_ref_numbers
0,북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?,[올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정...,북태평양 기단과 오호츠크해 기단이 만나 형성되는 장마전선은 한반도에 약 4~5주 동...,"[1, 5]"
1,지능형 생산자동화 기반기술을 개발중인 스타트업은?,"[네스트(32억달러), 비츠(30억달러), 오큘러스VR(20억달러). 지난해 구글 ...",지능형 생산자동화 기반기술을 개발 중인 스타트업으로는 마키나락스가 있습니다. 마키나...,[5]
2,개막전에서 3안타 2실점을 기록해서 패한 선수는?,[그해 8월 맥그로는 아메리칸 어소시에이션의 볼티모어 오리올스와 함께 자신의 메이저...,개막전에서 3안타 2실점을 기록하여 패한 선수는 사이타마 세이부 라이온스의 와쿠이 ...,[5]
3,컵라면 매출에서 불닭볶음면을 이긴 상품은?,[‘짜파구리’로 시작된 ‘나만의 레시피’ 열풍이 올해 식품시장을 주도한 것으로 나타...,컵라면 매출에서 불닭볶음면을 이긴 상품은 세븐일레븐에서 판매된 '교동반점 짬뽕'입니...,[2]
4,에티하드 웰니스 프로그램의 일환으로 위생에 관한 정보를 제공하는 것은 누구인가?,[위워크(WeWork)는 코로나19가 확산되는 가운데 보다 안전한 업무 환경 조성과...,에티하드 웰니스 프로그램의 일환으로 위생에 관한 정보를 제공하는 것은 에티하드항공의...,[5]


In [22]:
len(extracted_ref_numbers)

10